# EchoAlign: linear probes and transfer learning

This notebook demonstrates how to use EchoAlign's **832-dimensional fixed representation** (64 principal components per anatomical slot + global slot) to fit linear probes on downstream tasks (classification or regression).

The PCA projection matrices were fitted once on the private cohort's training studies and shipped in the weights file, so every user gets the same representation.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

from echoalign import EchoAlign
from eval.analysis.probes import classification_probe, regression_probe

# Load model to access PCA matrices and architecture
weights_path = '../weights/EchoAlign.pt'
model = EchoAlign(weights_path)
print('EchoAlign loaded successfully. PCA levels:', list(model.pca.keys()))


## 1. Extract 832-D Representation from Study Embeddings

When embedding studies with `model.encode_study([clips])`, the output is 13 vectors of 512-D `[13, 512]`.
`model.pca_embedding(study)` reduces this to an 832-D vector ready for standard Scikit-Learn linear models.

In [ ]:
# Synthetic study embeddings demonstrating the probe protocol: 60 studies
rng = np.random.default_rng(42)
synthetic_study_embeddings = rng.normal(size=(60, 13, 512))

# Extract 832-D representation
features = model.pca_embedding(synthetic_study_embeddings, level='study')
print('Extracted probe feature shape:', features.shape)


## 2. Fit Standard Paper-Protocol Classification Probe

The standardized probe protocol evaluated in the paper:
1. Fits a StandardScaler on TRAIN.
2. Tests L2 logistic regression across regularization parameter $C \in \{10^{-4}, 10^{-3}, 10^{-2}, 0.1, 1, 10, 100\}$.
3. Selects $C$ on VAL by AUROC.
4. Refits on TRAIN + VAL and scores TEST.

In [ ]:
# Split into train (30), val (15), test (15)
x_train, x_val, x_test = features[:30], features[30:45], features[45:]

# Example binary target (e.g. classification of a pathology)
y_train = rng.integers(0, 2, size=30)
y_val = rng.integers(0, 2, size=15)
y_test = rng.integers(0, 2, size=15)

# Run probe
pred_test_prob, selection = classification_probe(x_train, y_train, x_val, y_val, x_test)
print('Selected C:', selection['C'], 'from:', selection['C_source'])
print('Validation AUROC:', round(selection['validation_auroc'], 4) if selection['validation_auroc'] else 'N/A')
print('Test predictions (first 5 probabilities):', np.round(pred_test_prob[:5], 3))
